# Aging Interpolation (Leave-One-Aging-Out)

> **Authors:** Davide Corso, Marco Soldani  
> **Context:** Analysis of Galvanostatic Electrochemical Impedance Spectroscopy (GEIS) data on Lithium-Ion batteries (LiCoO₂).
>
**Expected Result:** Exclude an **entire Aging Level** (all 8 Nyquist plots across different temperatures × 5 SOC) and predict it using the other 4 Aging levels. By default, we exclude **Aging 2**, i.e. the central one: training is performed on Aging 0, 1, 3, and 4 → prediction is made on the 40 missing Nyquist diagrams.

This simulates a real-world scenario where we interpolate the battery behavior at an **unseen aging level that has never been measured**.

---

### Difference with the other previos LOO notebook (1: LOO)

| | Leave-One-Out (NB1) | Leave-One-Aging-Out (NB3) |
|---|---|---|
| **Excluded data** | 1 Aging × Temperature combination (~245 samples) | 1 full Aging level (~1960 samples, 40 Nyquist plots) |
| **Training** | 39 combinations | 32 combinations (4 Aging × 8 Temperature) |
| **Challenge** | Interpolate one missing plot among known neighbors | Interpolate an entire unseen aging level |
| **Available information** | Same Aging at other temperatures + same temperature at other aging levels | Only other aging levels, **no data from the excluded Aging** |

This is a **significantly more difficult task**: the model has no direct information about the excluded aging level and must truly *interpolate across aging states*.

# 1. Setup

In [1]:
import sys
assert sys.version_info >= (3, 5), "Python 3.5+ is required"

# --- STANDARD LIBRARIES ---
import warnings
import time

# --- SCIENTIFIC LIBRARIES ---
import numpy as np
import pandas as pd

# --- VISUALIZATION ---
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

# --- MACHINE LEARNING ---
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    BaggingRegressor,
    ExtraTreesRegressor,
)
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score,
)

# --- CONFIGURATION ---
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
mpl.rc("axes", labelsize=13)
mpl.rc("xtick", labelsize=11)
mpl.rc("ytick", labelsize=11)
mpl.rc("legend", fontsize=10)
mpl.rc("figure", dpi=120)

# --- CONSTANTS ---
TEMP_COLORS = {
    20.0: "#1b9e77",
    22.5: "#d95f02",
    25.0: "#7570b3",
    27.5: "#e7298a",
    30.0: "#66a61e",
    35.0: "#e6ab02",
    40.0: "#a6761d",
    47.5: "#1f3a93",
}

AGING_LABELS = {
    0: "Aging 0 (Fresh)",
    1: "Aging 1",
    2: "Aging 2",
    3: "Aging 3",
    4: "Aging 4 (Aged)",
}

SOC_MARKERS = {
    0: "o",
    1: "s",
    2: "^",
    3: "D",
    4: "v",
}

AGINGS = [0, 1, 2, 3, 4]

# --- ENVIRONMENT INFO ---
def print_environment_info():
    import sklearn

    print("Setup completed")
    print(
        f"Python {sys.version_info.major}.{sys.version_info.minor} | "
        f"NumPy {np.__version__} | "
        f"Pandas {pd.__version__} | "
        f"Scikit-learn {sklearn.__version__}"
    )


print_environment_info()

Setup completed
Python 3.12 | NumPy 1.26.4 | Pandas 2.3.3 | Scikit-learn 1.7.1


# 2. Data Loading

Loading the cleaned dataset from the CSV file generated from the original `.mat` data.

In [3]:
# --- LOAD CSV ---

from pathlib import Path

file_path = Path("..") / "data" / "processed" / "batteries_cleaned_dataset.csv"
df = pd.read_csv(file_path)

# Rename Epoch column to Aging (correct terminology)
if "Epoch" in df.columns:
    df = df.rename(columns={"Epoch": "Aging"})

# Sort for better DataFrame readability
df = df.sort_values(
    ["Aging", "Temperature", "SOC", "Frequency"]
).reset_index(drop=True)

TEMPS_SORTED = sorted(df["Temperature"].unique())
SOCS_SORTED = sorted(df["SOC"].unique())

# --- DATASET SUMMARY ---
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Aging levels: {sorted(df['Aging'].unique())}")
print(f"Temperatures: {TEMPS_SORTED} °C")
print(f"SOCs: {SOCS_SORTED}")
print(
    f"Frequency range: {df['Frequency'].min():.2f} – {df['Frequency'].max():.0f} Hz"
)
print(
    "Combinations (Aging × Temperature × SOC): "
    f"{df.groupby(['Aging','Temperature','SOC']).ngroups}"
)

df.head()

Dataset loaded: 9,805 rows × 6 columns
Aging levels: [0, 1, 2, 3, 4]
Temperatures: [20.0, 22.5, 25.0, 27.5, 30.0, 35.0, 40.0, 47.5] °C
SOCs: [0, 1, 2, 3, 4]
Frequency range: 0.10 – 10002 Hz
Combinations (Aging × Temperature × SOC): 200


,Aging,Temperature,SOC,Frequency,Z_real,Z_imag
0,0,20.0,0,0.126502,6.089266,0.467721
1,0,20.0,0,0.160118,6.057238,0.388003
2,0,20.0,0,0.202344,6.022578,0.322534
3,0,20.0,0,0.255913,5.991356,0.273686
4,0,20.0,0,0.323687,5.969386,0.238370
